# Cross-Strategy Comparison Dashboard

Runs every strategy in `source.STRATEGY_REGISTRY` across its native
`(group, timeframe, asset)` grid using each strategy's documented baseline
parameters, then renders six dashboard panels comparing them side-by-side.

**Sections**
1. Setup
2. Run all strategies (cached to parquet)
3. Leaderboard — top rows by selected metric
4. Per-asset heatmap — strategies × assets, one panel per (group, tf)
5. Equity overlay — top-N strategies on one (group, tf, asset)
6. Distribution plot — robustness of each strategy across cells
7. Return correlation — diversification candidates on a benchmark asset
8. Group breakdown — per-strategy aggregate per group
9. Takeaways

See `DocumentationVault/general/Strategy_Comparison_Dashboard.md` for the
full interpretation guide and how to extend the registry.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from source import (
    STRATEGY_REGISTRY,
    run_all_strategies, daily_returns_from_equity,
    plot_strategy_leaderboard, plot_strategy_heatmap,
    plot_strategy_equity_overlay, plot_strategy_metric_distribution,
    plot_strategy_return_correlation, plot_strategy_group_breakdown,
    cpu_count,
)

N_JOBS = "auto"
CACHE_DIR = REPO_ROOT / "comparison" / ".cache"
print(f"Parallelism budget: n_jobs={N_JOBS!r}  cpu_count={cpu_count()}")
print(f"Cache dir: {CACHE_DIR.relative_to(REPO_ROOT)}")
print(f"Strategies registered: {len(STRATEGY_REGISTRY)}")

In [ ]:
# Quick visibility on what we're about to run.
registry_df = pd.DataFrame([
    {
        "strategy": r.name,
        "notebook": r.notebook_id,
        "groups": ", ".join(r.group_timeframes),
        "timeframes": ", ".join(sorted({tf for tfs in r.group_timeframes.values() for tf in tfs})),
    }
    for r in STRATEGY_REGISTRY
])
registry_df

### Data availability

The runner silently skips groups with no CSV files under `data/<group>/`.
If you see fewer rows than expected in the leaderboard, the cell below tells
you which groups were absent (the Crypto group is expected to be missing in
the standard checkout — strategies that target Crypto would surface a warning here).

In [ ]:
from source import discover_datasets

metas = discover_datasets(REPO_ROOT / "data")
present_groups = sorted({m.group for m in metas.values() if m.group})
targeted = sorted({g for r in STRATEGY_REGISTRY for g in r.group_timeframes})
missing = sorted(set(targeted) - set(present_groups))
print(f"Groups present: {present_groups}")
print(f"Groups any strategy targets: {targeted}")
if missing:
    print(f"[warning] Missing data for: {missing} — strategies targeting these groups will produce zero rows.")

## 2. Run all strategies

First call populates `CACHE_DIR/` with one parquet triple per strategy
(`<slug>__<fingerprint>__{metrics,equity,trades}.parquet`). Re-running
this notebook reuses the cache; pass `force=True` or delete the cache
directory to invalidate. Changing a strategy's `base_kwargs` /
`b3_overrides` / `group_timeframes` in `source/comparison.py` rotates
the fingerprint and invalidates just that strategy.

In [ ]:
metrics_df, equity_df, trades_df = run_all_strategies(
    REPO_ROOT / "data",
    cache_dir=CACHE_DIR,
    n_jobs=N_JOBS,
    progress=True,
    force=False,
)
print(f"\nmetrics_df: {metrics_df.shape}")
print(f"equity_df:  {equity_df.shape}")
print(f"trades_df:  {trades_df.shape}")
print(f"Strategies with results: {sorted(metrics_df['strategy'].unique())}")

## 3. Leaderboard

Top rows of `(strategy, group, tf, asset)` by daily Sharpe (default).
Edit `METRIC` to rank by another column — `total_pnl`, `profit_factor`,
`expectancy`, `win_rate`, `max_drawdown` (set `ascending=True` for
drawdown so the *least bad* sort first).

In [ ]:
METRIC = "sharpe_daily"
TOP_N = 25

summary_cols = ["strategy", "group", "tf", "asset",
                "num_trades", "sharpe_daily", "profit_factor",
                "win_rate", "max_drawdown", "total_pnl"]
leaderboard = (metrics_df[summary_cols]
               .dropna(subset=[METRIC])
               .sort_values(METRIC, ascending=False)
               .head(TOP_N)
               .reset_index(drop=True))
leaderboard.style.format({
    "sharpe_daily": "{:+.2f}", "profit_factor": "{:.2f}",
    "win_rate": "{:.1%}", "max_drawdown": "{:,.1f}",
    "total_pnl": "{:,.1f}",
}) if not leaderboard.empty else leaderboard

In [ ]:
fig = plot_strategy_leaderboard(metrics_df, metric=METRIC, top_n=TOP_N)
plt.show()

## 4. Per-asset heatmap

One ``strategy × asset`` panel per ``(group, timeframe)``. Sharpe is
color-centered at zero so green = positive, red = negative. Blank cells
mean the strategy doesn't include that timeframe in its registry entry.

In [ ]:
fig = plot_strategy_heatmap(metrics_df, metric=METRIC)
plt.show()

## 5. Equity overlay (top-N on one cell)

Pick a `(group, tf, asset)` and the top-`N` strategies by `RANK_BY`
are overlaid on the same chart so you can see whose curve actually
compounds vs. just oscillates. The asset suggestion below is the cell
with the most strategies producing trades (i.e. the most visually
informative comparison).

In [ ]:
if not equity_df.empty:
    coverage = (equity_df[["strategy", "group", "tf", "asset"]]
                .drop_duplicates()
                .groupby(["group", "tf", "asset"]).size()
                .sort_values(ascending=False))
    suggested = coverage.head(8).reset_index().rename(columns={0: "n_strategies"})
    display(suggested)
    SEL_GROUP, SEL_TF, SEL_ASSET = coverage.index[0]
else:
    SEL_GROUP = SEL_TF = SEL_ASSET = None
    print("No equity data available.")

In [ ]:
TOP_N_OVERLAY = 5
RANK_BY = "sharpe_daily"
if SEL_GROUP is not None:
    fig = plot_strategy_equity_overlay(
        equity_df, metrics_df,
        group=SEL_GROUP, tf=SEL_TF, asset=SEL_ASSET,
        top_n=TOP_N_OVERLAY, rank_by=RANK_BY,
    )
    plt.show()
else:
    print("Skipped — no data.")

## 6. Metric distribution per strategy

Box plot of the metric across every `(tf, asset)` cell, split by group.
Wide boxes mean the strategy lives or dies by the cell it lands on
(possible overfitting). Tight boxes near zero mean the strategy is
consistent but unremarkable. Tight high boxes are the holy grail.
Strategies are sorted by median descending.

In [ ]:
fig = plot_strategy_metric_distribution(metrics_df, metric=METRIC, by_group=True)
plt.show()

## 7. Return correlation on a benchmark asset

Pairwise correlation of *daily PnL* for strategies that produced trades
on the selected benchmark cell. Strategies with correlation close to 1
are doing the same thing (and don't diversify each other); near-zero or
negative pairs are diversification candidates.

Daily PnL is computed from the equity points series via diff-on-resample,
so days with no trades read as zero. Use the same `SEL_GROUP / SEL_TF /
SEL_ASSET` selected above, or set explicit values.

In [ ]:
if SEL_GROUP is not None:
    daily_returns = daily_returns_from_equity(equity_df)
    fig = plot_strategy_return_correlation(
        daily_returns, group=SEL_GROUP, tf=SEL_TF, asset=SEL_ASSET
    )
    plt.show()
else:
    print("Skipped — no data.")

## 8. Group breakdown

Per-strategy aggregate of the metric inside each group (mean across the
strategy's TFs × assets in that group). Strategies that perform well in
both groups *travel* — they aren't single-market artefacts.

In [ ]:
fig = plot_strategy_group_breakdown(metrics_df, metric=METRIC, agg="mean")
plt.show()

## 9. Takeaways

Free-form interpretation cell — fill in after running. Suggested prompts:

* **Which strategies make the leaderboard at all?** (Sharpe ≥ 0.5 with
  num_trades ≥ 50 is a reasonable significance floor.)
* **Which strategies travel across groups?** (look at Section 8 — both
  forex *and* b3 bars green is the green flag.)
* **Which strategies depend on one asset?** (Section 6 — wide boxes
  with a single positive outlier.)
* **Diversification candidates?** (Section 7 — low pairwise correlation
  + both strategies positive Sharpe.)
* **Open follow-ups:** any cell that should be promoted to a dedicated
  WFO run, or a strategy whose baseline params look mis-calibrated for
  the group/tf in question.